<a href="https://colab.research.google.com/github/ethan-wood-uk/APRR-Concession-Model/blob/main/APRR_Concession_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# APRR Motorway Concession — Project Finance Model

## Background

APRR (Autoroutes Paris-Rhin-Rhône) is one of France's largest motorway concession
operators, holding the right to collect tolls on approximately 2,323 km of motorway
in eastern France until November 2035, when the network reverts to the French State.

In February 2006, a consortium of Macquarie and Eiffage acquired 81.5% of APRR from
the French government as part of a broader motorway privatisation programme.

This model is built as if at the point of that acquisition — assessing what return
an investor could expect to earn over the remaining concession life, given assumptions
about traffic growth, inflation, and financing structure.

## Section 1 — Assumptions and Inputs

All model inputs are defined here in one place. Inputs grounded in published data or
market convention are noted as such. Forward assumptions are anchored to historical
ranges where possible. Items marked DATA NEEDED require extraction from the 2005 APRR
annual report before the model produces meaningful outputs.

In [4]:
!pip install numpy_financial

In [5]:
import numpy as np
import pandas as pd
import numpy_financial as npf
import matplotlib.pyplot as plt
from datetime import date

# ── TIMING ──────────────────────────────────────────────────
ACQUISITION_DATE    = date(2006, 2, 1)   # Real transaction date
CONCESSION_END_DATE = date(2035, 11, 30) # Confirmed: concession agreement
PERIODS_PER_YEAR    = 2                  # Semi-annual model

concession_years = (CONCESSION_END_DATE - ACQUISITION_DATE).days / 365.25
N_PERIODS = int(concession_years * PERIODS_PER_YEAR)

print(f"Concession life: {concession_years:.1f} years | Semi-annual periods: {N_PERIODS}")

Concession life: 29.8 years | Semi-annual periods: 59


In [6]:
# ── NETWORK AND TRAFFIC ──────────────────────────────────────
# Network length confirmed from APRR published data (APRR + AREA combined)
NETWORK_LENGTH_KM = 2323

# Initial volumes: to be extracted from 2005 APRR annual report
INITIAL_LIGHT_VEH_PER_DAY = 600000   # DATA NEEDED — 2005 annual report
INITIAL_HEAVY_VEH_PER_DAY = 80000    # DATA NEEDED — 2005 annual report

# Growth rates: forward assumptions anchored to APRR pre-2006 historical growth
# Light vehicles historically ~1.5-3.0% p.a.; heavy ~0.5-2.0% p.a.
# Monte Carlo will vary both across plausible ranges
LIGHT_VEH_GROWTH_RATE_PA = 0.020  # Base case: 2.0% p.a.
HEAVY_VEH_GROWTH_RATE_PA = 0.015  # Base case: 1.5% p.a.

In [7]:
# ── TOLLS, OPEX AND INFLATION ────────────────────────────────
# Toll rates: published and regulated; to be extracted from 2005 tariff schedules
# Simplified to one weighted average rate per vehicle class
INITIAL_TOLL_LIGHT_EUR_PER_VEH = 5.5  # DATA NEEDED — 2005 tariff schedule
INITIAL_TOLL_HEAVY_EUR_PER_VEH = 18  # DATA NEEDED — 2005 tariff schedule

# Opex: fixed per km and variable per vehicle; to be extracted from 2005 accounts
# Major maintenance ~€150m p.a. across network (APRR published)
FIXED_OPEX_EUR_PER_KM_PA  = 64572   # DATA NEEDED ~€150m total / 2,323 km
VARIABLE_OPEX_EUR_PER_VEH = 0.3   # DATA NEEDED — 2005 accounts

# Inflation: French CPI ~1.7-1.9% in 2006; base case uses ECB 2% target
# Tolls and opex both assumed to escalate at CPI — simplification
# Monte Carlo will vary inflation across plausible ranges
CPI_RATE_PA             = 0.020  # Base case: anchored to 2006 ECB target
TOLL_ESCALATION_RATE_PA = CPI_RATE_PA
OPEX_INFLATION_RATE_PA  = CPI_RATE_PA

# Working capital: standard terms for a regulated infrastructure asset
RECEIVABLES_DAYS = 30  # Electronic toll collection — short cycle
PAYABLES_DAYS    = 60  # Standard supplier terms

In [8]:
# ── TAX, FINANCING AND VALUATION ─────────────────────────────
# Tax: French corporate rate 2006 was 33.33%
# Depreciation: acquisition price treated as concession intangible,
# amortised straight-line over remaining concession life (simplification —
# full purchase price allocation not modelled; noted in README limitations)
CORPORATE_TAX_RATE = 0.3333
DEPRECIATION_YEARS = concession_years

# Debt: terms consistent with 2006 European investment-grade infra acquisition
# APRR was A-rated at acquisition; interest rate anchored to 2006 Euribor + spread
DSCR_MINIMUM     = 1.35   # Market convention for toll road acquisition
MAX_DEBT_TO_COST = 0.80   # 80% LTV cap — market convention
DEBT_TENOR_YEARS = 15     # Standard tenor for investment-grade infra
INTEREST_RATE_PA = 0.045  # ~4.5% all-in (Euribor ~3.5-4% + spread)
ARRANGEMENT_FEE  = 0.010  # 1% upfront fee — market convention

# Valuation: core brownfield infra target return in 2006 was 8-10% equity IRR
# Model solves for acquisition price at target IRR and validates against
# actual 2006 transaction enterprise value
TARGET_EQUITY_IRR     = 0.080  # Conservative floor: 8.0%
ACQUISITION_PRICE_EUR = 0      # DATA NEEDED — 2006 transaction EV

In [9]:
# ── INPUT SUMMARY ────────────────────────────────────────────
print(f"{'APRR ACQUISITION MODEL — INPUT SUMMARY':^55}")
print("=" * 55)
print(f"Concession life:        {concession_years:.1f} yrs | Periods: {N_PERIODS}")
print(f"Network:                {NETWORK_LENGTH_KM:,} km")
print(f"Light veh/day (2005):   {INITIAL_LIGHT_VEH_PER_DAY:,}  [DATA NEEDED]")
print(f"Heavy veh/day (2005):   {INITIAL_HEAVY_VEH_PER_DAY:,}  [DATA NEEDED]")
print(f"Light growth:           {LIGHT_VEH_GROWTH_RATE_PA:.1%} p.a.")
print(f"Heavy growth:           {HEAVY_VEH_GROWTH_RATE_PA:.1%} p.a.")
print(f"Light toll rate:        €{INITIAL_TOLL_LIGHT_EUR_PER_VEH:.2f}/veh  [DATA NEEDED]")
print(f"Heavy toll rate:        €{INITIAL_TOLL_HEAVY_EUR_PER_VEH:.2f}/veh  [DATA NEEDED]")
print(f"Fixed opex:             €{FIXED_OPEX_EUR_PER_KM_PA:,.0f}/km/yr  [DATA NEEDED]")
print(f"Variable opex:          €{VARIABLE_OPEX_EUR_PER_VEH:.2f}/veh  [DATA NEEDED]")
print(f"CPI / escalation:       {CPI_RATE_PA:.1%} p.a.")
print(f"Receivables / payables: {RECEIVABLES_DAYS}d / {PAYABLES_DAYS}d")
print(f"Corporate tax:          {CORPORATE_TAX_RATE:.2%}")
print(f"Depreciation life:      {DEPRECIATION_YEARS:.1f} yrs")
print(f"DSCR minimum:           {DSCR_MINIMUM:.2f}x")
print(f"Max LTV:                {MAX_DEBT_TO_COST:.0%}")
print(f"Debt tenor:             {DEBT_TENOR_YEARS} yrs")
print(f"Interest rate:          {INTEREST_RATE_PA:.2%} p.a.")
print(f"Arrangement fee:        {ARRANGEMENT_FEE:.1%}")
print(f"Target equity IRR:      {TARGET_EQUITY_IRR:.1%}")
print(f"Acquisition price:      €{ACQUISITION_PRICE_EUR:,.0f}  [DATA NEEDED]")
print("=" * 55)

        APRR ACQUISITION MODEL — INPUT SUMMARY         
Concession life:        29.8 yrs | Periods: 59
Network:                2,323 km
Light veh/day (2005):   600,000  [DATA NEEDED]
Heavy veh/day (2005):   80,000  [DATA NEEDED]
Light growth:           2.0% p.a.
Heavy growth:           1.5% p.a.
Light toll rate:        €5.50/veh  [DATA NEEDED]
Heavy toll rate:        €18.00/veh  [DATA NEEDED]
Fixed opex:             €64,572/km/yr  [DATA NEEDED]
Variable opex:          €0.30/veh  [DATA NEEDED]
CPI / escalation:       2.0% p.a.
Receivables / payables: 30d / 60d
Corporate tax:          33.33%
Depreciation life:      29.8 yrs
DSCR minimum:           1.35x
Max LTV:                80%
Debt tenor:             15 yrs
Interest rate:          4.50% p.a.
Arrangement fee:        1.0%
Target equity IRR:      8.0%
Acquisition price:      €0  [DATA NEEDED]


## Section 2 — Operating Model

This section builds the core cash flow engine of the model. Starting from initial
traffic volumes and toll rates, it computes revenue, operating costs, and CFADS
(Cash Flow Available for Debt Service) across all 59 semi-annual periods to 2035.

In [10]:
# ── SECTION 2: OPERATING MODEL ───────────────────────────────
# ── 2.1 Timeline ─────────────────────────────────────────────

periods = np.arange(N_PERIODS)
days_per_period = 365.25 / 2
years_elapsed = periods / PERIODS_PER_YEAR

# ── 2.2 Inflation Index ───────────────────────────────────────
# Compounds CPI semi-annually from acquisition date
# Used to escalate both toll rates and opex each period

inflation_index = (1 + CPI_RATE_PA) ** years_elapsed

# ── 2.3 Traffic Volumes ───────────────────────────────────────
# Daily volumes compounded annually from initial 2005 figures
# Converted to per-period volumes using days_per_period

traffic_light = INITIAL_LIGHT_VEH_PER_DAY * (1 + LIGHT_VEH_GROWTH_RATE_PA) ** years_elapsed * days_per_period
traffic_heavy = INITIAL_HEAVY_VEH_PER_DAY * (1 + HEAVY_VEH_GROWTH_RATE_PA) ** years_elapsed * days_per_period

# ── 2.4 Inflated Toll Rates ───────────────────────────────────
# Real toll rates escalated by inflation index each period
# Simplified to one weighted average rate per vehicle class

toll_light = INITIAL_TOLL_LIGHT_EUR_PER_VEH * inflation_index
toll_heavy = INITIAL_TOLL_HEAVY_EUR_PER_VEH * inflation_index

In [11]:
# ── 2.5 Revenue ───────────────────────────────────────────────
# Total revenue per period = traffic × inflated toll rate
# Summed across light and heavy vehicle classes

revenue_light = traffic_light * toll_light
revenue_heavy = traffic_heavy * toll_heavy
total_revenue = revenue_light + revenue_heavy

In [12]:
# ── 2.6 Operating Expenditure ─────────────────────────────────
# Fixed opex: cost per km per year, converted to per-period, inflation-linked
# Variable opex: cost per vehicle, inflation-linked

opex_fixed    = FIXED_OPEX_EUR_PER_KM_PA * NETWORK_LENGTH_KM * (1 / PERIODS_PER_YEAR) * inflation_index
opex_variable = VARIABLE_OPEX_EUR_PER_VEH * (traffic_light + traffic_heavy) * inflation_index
total_opex    = opex_fixed + opex_variable

In [13]:
 # ── 2.7 EBITDA ────────────────────────────────────────────────
ebitda = total_revenue - total_opex

In [20]:
# ── 2.8 Working Capital ───────────────────────────────────────
# Change in net working capital each period
# Receivables increase as revenue grows — cash lags revenue slightly
# Payables increase as opex grows — cash lags payments slightly

receivables = total_revenue * (RECEIVABLES_DAYS / 365.25)
payables    = total_opex * (PAYABLES_DAYS / 365.25)
nwc         = receivables - payables
delta_nwc   = np.diff(nwc, prepend=nwc[0])

In [26]:
# ── 2.9 CFADS ─────────────────────────────────────────────────
# Cash Flow Available for Debt Service
# EBITDA less change in working capital
# Tax deducted in Section 3 after interest is known

cfads_pretax = ebitda - delta_nwc